# Mistral-7B (Unsloth LoRA) - Abstract Evaluator SFT

This notebook trains `mistralai/Mistral-7B-Instruct-v0.3` using Unsloth + LoRA on chat-format JSONL data, logs to Weights & Biases, evaluates each epoch, and reports ROUGE/BLEU/BERTScore on test.

In [ ]:
# If running first time, uncomment:
# !pip install -U unsloth transformers datasets trl peft accelerate bitsandbytes wandb evaluate rouge_score nltk bert_score scikit-learn

import os
import json
import random
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, DatasetDict

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ------------------------------
# Paths and experiment config
# ------------------------------
PROJECT_ROOT = r"C:\\Users\\hanib\\Desktop\\nlp_final_final_final_final_project\\Abstract-Evaluator"
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'sft')
os.makedirs(DATA_DIR, exist_ok=True)

# Preferred: provide these JSONL files directly
TRAIN_PATH = os.path.join(DATA_DIR, 'train.jsonl')
VAL_PATH = os.path.join(DATA_DIR, 'val.jsonl')
TEST_PATH = os.path.join(DATA_DIR, 'test.jsonl')

# Optional fallback: one combined JSONL with exactly 10k rows
COMBINED_PATH = os.path.join(DATA_DIR, 'all.jsonl')

MODEL_NAME = 'mistralai/Mistral-7B-Instruct-v0.3'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'mistral7b_unsloth_lora')
RUN_NAME = 'mistral7b-abstract-evaluator-unsloth-lora'

# Sequence and LoRA setup
MAX_SEQ_LENGTH = 1024
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05

# Mistral-7B has 32 transformer layers
# Train top 12 layers with LoRA -> freeze 20/32 = 62.5% (within your 50-70% target)
TOTAL_LAYERS = 32
TRAIN_TOP_K = 12
LAYERS_TO_TRANSFORM = list(range(TOTAL_LAYERS - TRAIN_TOP_K, TOTAL_LAYERS))

# For ~80GB VRAM this is usually safe with 4-bit + gradient checkpointing
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRAD_ACC_STEPS = 2
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 3
WARMUP_RATIO = 0.03

print('Will LoRA-train layers:', LAYERS_TO_TRANSFORM)

In [ ]:
# ------------------------------
# Dataset loading + split checks
# ------------------------------
def file_exists(p):
    return os.path.isfile(p) and os.path.getsize(p) > 0

if file_exists(TRAIN_PATH) and file_exists(VAL_PATH) and file_exists(TEST_PATH):
    data_files = {'train': TRAIN_PATH, 'validation': VAL_PATH, 'test': TEST_PATH}
    ds = load_dataset('json', data_files=data_files)
else:
    assert file_exists(COMBINED_PATH), (
        f'Missing split files and missing combined file: {COMBINED_PATH}. '
        'Provide train/val/test JSONL or all.jsonl in data/sft.'
    )
    full = load_dataset('json', data_files={'all': COMBINED_PATH})['all'].shuffle(seed=SEED)
    assert len(full) >= 10000, f'Need at least 10,000 rows, found {len(full)}'
    full = full.select(range(10000))
    ds = DatasetDict({
        'train': full.select(range(0, 8000)),
        'validation': full.select(range(8000, 9000)),
        'test': full.select(range(9000, 10000)),
    })

print(ds)
print('Train size:', len(ds['train']))
print('Val size:', len(ds['validation']))
print('Test size:', len(ds['test']))

# Basic format check
sample = ds['train'][0]
assert 'messages' in sample, 'Each row must contain `messages`'
assert isinstance(sample['messages'], list), '`messages` must be a list'
assert len(sample['messages']) >= 2, 'Need at least user + assistant messages'
print('Sample OK:', sample['messages'][0]['role'], '->', sample['messages'][1]['role'])

In [ ]:
# ------------------------------
# Weights & Biases
# ------------------------------
import wandb

# Option 1: set WANDB_API_KEY in environment before running
# Option 2: run wandb.login(key='...')
wandb.login()

wandb.init(
    project='abstract-evaluator-sft',
    name=RUN_NAME,
    config={
        'model': MODEL_NAME,
        'max_seq_length': MAX_SEQ_LENGTH,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_dropout': LORA_DROPOUT,
        'layers_to_transform': LAYERS_TO_TRANSFORM,
        'batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
        'grad_accumulation': GRAD_ACC_STEPS,
        'learning_rate': LEARNING_RATE,
        'epochs': NUM_EPOCHS,
        'train_rows': len(ds['train']),
        'val_rows': len(ds['validation']),
        'test_rows': len(ds['test'])
    }
)

In [ ]:
# ------------------------------
# Model + tokenizer (Unsloth)
# ------------------------------
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing='unsloth',
    bias='none',
    random_state=SEED,
    layers_to_transform=LAYERS_TO_TRANSFORM,
)

print('LoRA model prepared.')

In [ ]:
# ------------------------------
# Formatting for chat SFT
# ------------------------------
def formatting_prompts_func(examples):
    texts = []
    for msgs in examples['messages']:
        text = tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {'text': texts}

train_ds = ds['train'].map(formatting_prompts_func, batched=True)
val_ds = ds['validation'].map(formatting_prompts_func, batched=True)
test_ds = ds['test'].map(formatting_prompts_func, batched=True)

print(train_ds[0]['text'][:800])

In [ ]:
# ------------------------------
# Trainer setup
# ------------------------------
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    optim='adamw_8bit',
    logging_steps=10,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=['wandb'],
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_available(),
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field='text',
    args=train_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

trainer

In [ ]:
# ------------------------------
# Train
# ------------------------------
train_result = trainer.train()
print(train_result)

best_ckpt = trainer.state.best_model_checkpoint
print('Best checkpoint:', best_ckpt)

trainer.save_model(os.path.join(OUTPUT_DIR, 'final_lora_adapter'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'final_lora_adapter'))

In [ ]:
# ------------------------------
# Validation and test loss
# ------------------------------
val_metrics = trainer.evaluate(eval_dataset=val_ds)
test_loss_metrics = trainer.evaluate(eval_dataset=test_ds, metric_key_prefix='test')
print('Validation metrics:', val_metrics)
print('Test loss metrics:', test_loss_metrics)
wandb.log({**val_metrics, **test_loss_metrics})

In [ ]:



c,cld,cdl
















In [ ]:
# ------------------------------
# Save metrics + finish W&B
# ------------------------------
metrics_path = os.path.join(OUTPUT_DIR, 'final_metrics.json')
all_metrics = {}
all_metrics.update({k: float(v) for k, v in val_metrics.items() if isinstance(v, (int, float))})
all_metrics.update({k: float(v) for k, v in test_loss_metrics.items() if isinstance(v, (int, float))})
all_metrics.update(gen_metrics)

with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(all_metrics, f, indent=2)

print('Saved metrics to:', metrics_path)
wandb.finish()